# Steam Oyun Benzerlik Ağı – Poster (A3 PDF)
Beril Özdizdar · Sosyal Ağ Analizi dönem ödevi

Bu defter, ağın okunur etiketli ve lejantlı **A3 poster PDF**'ini üretir.
Sadece `steam-network.zip` gerekir. Toplulukları bu defter, 1. defterdeki yöntemle (Leiden, 10 tohum, en yüksek modülerlik) yeniden hesaplar.
Hücreleri sırayla çalıştır; yükleme penceresi çıkınca zip'i seç.

## 0. Kurulum ve veri

In [ ]:
!pip -q install adjustText igraph leidenalg

In [ ]:
import os, glob, subprocess, textwrap, warnings
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.collections import LineCollection
from google.colab import files
warnings.filterwarnings("ignore")

ROOT = "/content"
def bul(desen):
    return glob.glob(f"{ROOT}/**/{desen}", recursive=True)

def zip_iste(ad_deseni, mesaj, hedef):
    if not glob.glob(f"{ROOT}/{ad_deseni}"):
        print(mesaj)
        files.upload()
    for z in glob.glob(f"{ROOT}/{ad_deseni}"):
        subprocess.run(["unzip", "-q", "-o", z, "-d", hedef])

if not bul("processed/edges_game_game.csv"):
    zip_iste("steam-network*.zip", "Açılan pencereden steam-network.zip dosyasını seç:", ROOT)

gg_yol = bul("processed/edges_game_game.csv")
com_yol = bul("topluluk_atamalari.csv")
assert gg_yol, "Dosyalar bulunamadı: steam-network.zip yüklenmiş olmalı."
OUT3 = f"{ROOT}/sonuclar3"; os.makedirs(OUT3, exist_ok=True)
print("Veri bulundu.")

## 1. Ağı hazırla (okunurluk için filtreli görünüm)

In [ ]:
N_GOSTER = 300      # posterde gösterilecek en çok değerlendirilen oyun sayısı
ESIK = 0.65         # bu benzerliğin altındaki bağlantılar çizilmez

gg = pd.read_csv(gg_yol[0], encoding="utf-8-sig")
if com_yol:
    com = pd.read_csv(com_yol[0], encoding="utf-8-sig")
else:
    import json, igraph as ig, leidenalg
    gt = pd.read_csv(bul("processed/edges_game_tag.csv")[0], encoding="utf-8-sig")
    meta = {}
    for line in open(bul("raw/appdetails.jsonl")[0], encoding="utf-8"):
        if line.strip():
            d = json.loads(line)
            meta[d["appid"]] = (d.get("name", ""), d.get("positive", 0) + d.get("negative", 0))
    G = nx.Graph()
    G.add_nodes_from(gt["appid"].unique())
    G.add_weighted_edges_from(gg[["appid_a", "appid_b", "benzerlik"]].itertuples(index=False, name=None))
    nodes = list(G.nodes()); idx = {n: i for i, n in enumerate(nodes)}
    g_ig = ig.Graph(n=len(nodes), edges=[(idx[u], idx[v]) for u, v in G.edges()])
    g_ig.es["weight"] = [d["weight"] for _, _, d in G.edges(data=True)]
    en_iyi = None
    for sd in range(10):
        part = leidenalg.find_partition(g_ig, leidenalg.ModularityVertexPartition,
                                        weights="weight", n_iterations=-1, seed=sd)
        if en_iyi is None or part.modularity > en_iyi.modularity:
            en_iyi = part
    print(f"Leiden: modülerlik {en_iyi.modularity:.4f}, {len(set(en_iyi.membership))} topluluk")
    com = pd.DataFrame({"appid": nodes, "topluluk": en_iyi.membership,
                        "oyun": [meta.get(n, ("", 0))[0] for n in nodes],
                        "degerlendirme": [meta.get(n, ("", 0))[1] for n in nodes]})

sizes = com["topluluk"].value_counts()
top5 = list(sizes.index[:5])
ISIM = ["Aksiyon-RPG, açık dünya", "Strateji, simülasyon", "FPS, çok oyunculu nişancı",
        "Korku, atmosferik macera", "Bağımsız, 2D, parti oyunları"]
KISA = ["Aksiyon-RPG", "Strateji ve simülasyon", "FPS ve nişancı", "Korku ve macera", "Bağımsız ve 2D"]
RENK = ["#66c0f4", "#a3cf06", "#f5a524", "#e0609b", "#b18cff"]
GRI = "#5b6676"
renk_of = {c: RENK[i] for i, c in enumerate(top5)}

sel = com.nlargest(N_GOSTER, "degerlendirme")
selset = set(sel["appid"])
e = gg[(gg["benzerlik"] >= ESIK) & gg["appid_a"].isin(selset) & gg["appid_b"].isin(selset)]
H = nx.Graph()
H.add_weighted_edges_from(e[["appid_a", "appid_b", "benzerlik"]].itertuples(index=False, name=None))
info = com.set_index("appid")

# Topluluk merkezli yerleşim: her topluluk kendi kümesinde, kümeler çember üzerinde
merkez = {c: (1.3 * np.cos(2 * np.pi * i / 5 + np.pi / 2), np.sin(2 * np.pi * i / 5 + np.pi / 2)) for i, c in enumerate(top5)}
merkez[-1] = (0.0, 0.0)
grp = {}
for n in H.nodes():
    c = info.loc[n, "topluluk"]
    grp.setdefault(c if c in merkez else -1, []).append(n)
pos, yaricap = {}, {}
for c, ns in grp.items():
    sub = H.subgraph(ns)
    p = nx.spring_layout(sub, weight="weight", seed=42, iterations=200) if len(ns) > 1 else {ns[0]: np.zeros(2)}
    ext = np.abs(np.array(list(p.values()))).max() or 1
    r = min(0.42, 0.07 * np.sqrt(len(ns)) + 0.1)
    for n, v in p.items():
        pos[n] = (merkez[c][0] + r * v[0] / ext, merkez[c][1] + r * v[1] / ext)
    yaricap[c] = r
print(f"Gösterilen: {H.number_of_nodes()} oyun, {H.number_of_edges()} bağlantı (tüm ağ: {len(com)} oyun, {len(gg)} bağlantı)")

## 2. A3 posteri çiz

In [ ]:
try:
    from adjustText import adjust_text
except Exception:
    adjust_text = None

BG, FG, MUT, PANEL = "#0f141b", "#e8eaed", "#9aa4b2", "#172231"
W, Hh = 11.69, 16.54     # A3 dikey (inç)

nodes = list(H.nodes())
xy = np.array([pos[n] for n in nodes])
rev = np.array([info.loc[n, "degerlendirme"] for n in nodes], float)
lr = np.log10(rev + 1)
boyut = 18 + 320 * ((lr - lr.min()) / (lr.max() - lr.min() + 1e-9)) ** 1.6
renkler = [renk_of.get(info.loc[n, "topluluk"], GRI) for n in nodes]

fig = plt.figure(figsize=(W, Hh), facecolor=BG)
fig.text(0.06, 0.965, "Steam Oyun Benzerlik Ağı", color="#f2f4f7", fontsize=44, fontweight="bold", va="top")
fig.text(0.06, 0.923, "Oyunlar etiket benzerliğine göre beş büyük türde toplanıyor",
         color="#66c0f4", fontsize=20, va="top")

# --- ağ paneli
ax = fig.add_axes([0.03, 0.36, 0.94, 0.54]); ax.set_facecolor(BG); ax.axis("off")
ic, dis = [], []
for u, v in H.edges():
    (ic if info.loc[u, "topluluk"] == info.loc[v, "topluluk"] else dis).append((pos[u], pos[v]))
ax.add_collection(LineCollection(dis, colors="#6f8fb0", linewidths=0.5, alpha=0.10, zorder=1))
ax.add_collection(LineCollection(ic, colors="#9fb8d0", linewidths=0.5, alpha=0.22, zorder=1))
ax.scatter(xy[:, 0], xy[:, 1], s=boyut, c=renkler, edgecolors=BG, linewidths=0.6, alpha=0.95, zorder=3)
ax.set_xlim(xy[:, 0].min() - 0.15, xy[:, 0].max() + 0.15)
ax.set_ylim(xy[:, 1].min() - 0.15, xy[:, 1].max() + 0.25)
ax.set_aspect("equal", adjustable="datalim")

en_populer = np.argsort(-rev)[:14]
metinler = []
for i, c in enumerate(top5):
    if c in yaricap:
        ax.text(merkez[c][0], merkez[c][1] + yaricap[c] + 0.06, KISA[i], color=RENK[i], fontsize=18,
                fontweight="bold", ha="center", va="bottom", zorder=6,
                path_effects=[pe.withStroke(linewidth=4, foreground=BG)])
for i in en_populer:
    ad = str(info.loc[nodes[i], "oyun"])
    ad = ad if len(ad) <= 24 else ad[:22] + "…"
    metinler.append(ax.text(xy[i, 0], xy[i, 1], ad, color="#f2f4f7", fontsize=11, fontweight="bold", zorder=5,
                            path_effects=[pe.withStroke(linewidth=3, foreground=BG)]))
if adjust_text is not None:
    adjust_text(metinler, ax=ax, arrowprops=dict(arrowstyle="-", color=MUT, lw=0.6))

# --- lejant
lg = fig.add_axes([0.04, 0.285, 0.92, 0.07]); lg.axis("off"); lg.set_xlim(0, 1); lg.set_ylim(0, 1)
ogeler = [(RENK[i], f"{ISIM[i]} · {int(sizes[top5[i]])} oyun") for i in range(5)]
ogeler.append((GRI, "Diğer küçük gruplar"))
for k, (r, t) in enumerate(ogeler):
    x, y = (k % 3) * 0.34, 0.72 if k < 3 else 0.2
    lg.scatter([x + 0.008], [y], s=300, color=r)
    lg.text(x + 0.03, y, t, va="center", color=FG, fontsize=13)

# --- bilgi panelleri
def panel(x0, baslik, govde, renk):
    p = fig.add_axes([x0, 0.075, 0.29, 0.195]); p.set_facecolor(PANEL)
    p.set_xticks([]); p.set_yticks([])
    for s in p.spines.values(): s.set_color("#2b4258")
    p.text(0.06, 0.94, baslik, transform=p.transAxes, color=renk, fontsize=17, fontweight="bold", va="top")
    p.text(0.06, 0.80, govde, transform=p.transAxes, color=FG, fontsize=13, va="top", linespacing=1.35)

def madde(metin, w=33):
    return "\n".join(textwrap.fill("• " + m, w, subsequent_indent="  ") for m in metin)

panel(0.04, "Nasıl okunur?", madde([
    "Nokta = oyun; büyüklük = değerlendirme sayısı.",
    "Renk = Leiden algoritmasının bulduğu topluluk.",
    f"Çizgi = etiket benzerliği ≥ {ESIK}.",
    f"Okunurluk için en çok değerlendirilen {H.number_of_nodes()} oyun gösterilir.",
]), "#66c0f4")
panel(0.355, "Ana bulgular", madde([
    "Kümelenme 0.54, modülerlik 0.63; rastgele ağlarda 0.09 ve 0.10.",
    "5 topluluk oyunların yaklaşık %70’ini kapsıyor.",
    "En olası 100 tahminin %43’ü yeni oyunlarda gerçekleşti (rastgele: %5).",
]), "#a3cf06")
panel(0.67, "Boş kalan fırsatlar", madde([
    "Female Protagonist + Sandbox: 0",
    "Great Soundtrack + PvE: 0",
    "Hack and Slash + Simulation: 1",
    "Management + Shooter: 1",
    "Exploration + Tactical: 2",
], 29) + "\n\n" + textwrap.fill("Sayı = kombinasyonu taşıyan oyun sayısı. Başarı garantisi değil.", 29), "#f5a524")

alt = ("Veri: SteamSpy, 25 Ekim 2026 · 2.000 oyun, 346 etiket · Yöntem: etiket oy vektörlerinin kosinüs benzerliği, "
       "Leiden topluluk tespiti, Adamic-Adar bağlantı tahmini · Beril Özdizdar · Analiz desteğinde Claude (Anthropic) kullanıldı.")
fig.text(0.06, 0.045, "\n".join(textwrap.wrap(alt, 125)), color=MUT, fontsize=10.5, va="top")

fig.savefig(f"{OUT3}/poster_A3.pdf", facecolor=BG)
fig.savefig(f"{OUT3}/poster_onizleme.png", dpi=80, facecolor=BG)
plt.show()

## 3. İndir

In [ ]:
import shutil
shutil.make_archive(f"{ROOT}/sonuclar3", "zip", OUT3)
files.download(f"{ROOT}/sonuclar3.zip")